<a href="https://colab.research.google.com/github/smaran19/resume-screening-and-hiring-prediction/blob/main/resume_screening_hiring_prediction_and_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="
    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
    padding: 25px;
    border-radius: 12px;
    border-left: 8px solid #00d4ff;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    margin-bottom: 20px;
    font-family: 'Inter', sans-serif;">
    
<h2 style="
    color: #00d4ff;
    margin: 0;
    text-transform: uppercase;
    letter-spacing: 2px;
    font-weight: 800;
    font-size: 1.8rem;">
    Resume Screening : Hiring Prediction and EDA
</h2>

<p style="
    color: #94a3b8;
    margin: 5px 0 0 0;
    font-size: 1rem;
    font-weight: 400;">
    Uncovering patterns in candidate performance and hiring probability.
</p>
</div>


# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">    Executive Summary 📋</span></div>
<p style="
    color: #94a3b8;
    margin: 5px 0 0 0;
    font-size: 1.2rem;
    font-weight: 400;">
    This notebook focuses on uncovering patterns in candidate performance and building a robust model using XGBoost for predicting hiring probability
</p>

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">Setup and Configuration ⚙️</span></div>

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')
pio.renderers.default = "iframe"
df = pd.read_csv('/kaggle/input/datasets/rhythmghai/resume-screening-dataset-200k-candidates/resume_dataset_200k_enhanced.csv')

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">    Exploratory Data Analysis (EDA)</span></div>

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">A. Target Variable Distribution: Hired vs. Not Hired</span>
<p style="
    color: #94a3b8;
    margin: 5px 0 0 0;
    font-size: 1.2rem;
    font-weight: 400;">
    A primary step is checking for class imbalance. Our dataset shows a significant number of candidates who were successfully hired (labeled '1') versus those who were not ('0').
</p>

In [ ]:
fig_target = px.pie(df, names='hired', title='Hiring Status Distribution',
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig_target.show()

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">B. Academic Performance & Hiring Correlation</span>
<p style="
    color: #94a3b8;
    margin: 5px 0 0 0;
    font-size: 1.2rem;
    font-weight: 400;">
    The relationship between cgpa and hired status is a critical indicator. Higher CGPAs generally cluster more densely around the "Hired" status, though it is not the sole determinant.
</p>

In [ ]:
fig_cgpa = px.box(df, x='hired', y='cgpa', color='hired',
             title='CGPA Distribution by Hiring Status',
             labels={'hired': 'Hired (1) vs Not (0)', 'cgpa': 'CGPA'})
fig_cgpa.show()

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">C. Impact of Education Level and University Tier</span>
<p style="
    color: #94a3b8;
    margin: 5px 0 0 0;
    font-size: 1.2rem;
    font-weight: 400;">
    Education level (Bachelors, Masters, PhD) and the university_tier (Tier 1, 2, or 3) are key categorical drivers.
    <li style="
        color: #94a3b8;
        margin: 5px 0 0 0;
        font-size: 1.2rem;
        font-weight: 400;">1. Education Level: Masters and PhD holders often show a higher probability of being hired in technical roles
    </li>
    <li style="
        color: #94a3b8;
        margin: 5px 0 0 0;
        font-size: 1.2rem;
        font-weight: 400;">2. University Tier: Tier 1 university candidates show a slightly higher hiring rate, though Tier 2 and 3 candidates are highly represented in the "Hired" pool if their skills_score is high.
    </li>
</p>

In [ ]:
fig_edu = px.sunburst(df, path=['education_level', 'university_tier', 'hired'],
                  title='Education Hierarchy & Hiring Outcomes')
fig_edu.show()

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">D. Professional Features: Skills</span>
<p style="
    color: #94a3b8;
    margin: 5px 0 0 0;
    font-size: 1.2rem;
    font-weight: 400;">
    Skills Score: This appears to be the strongest continuous predictor. Candidates with a skills_score above 15.0 have a markedly higher chance of being hired.
</p>

In [ ]:
fig_skills = px.histogram(df, x='skills_score', color='hired', marginal='box',
                   title='Impact of Skills Score on Hiring',
                   barmode='overlay', opacity=0.7)
fig_skills.show()

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">Insights for the Modeling Phase</span></div>
<p style="
    color: #94a3b8;
    margin: 5px 0 0 0;
    font-size: 1.2rem;
    font-weight: 400;">
    From this EDA, we can conclude:
    <li style="
        color: #94a3b8;
        margin: 5px 0 0 0;
        font-size: 1.2rem;
        font-weight: 400;">1. Feature Selection: skills_score, cgpa, and experience_years must be prioritized.
    <li style="
        color: #94a3b8;
        margin: 5px 0 0 0;
        font-size: 1.2rem;
        font-weight: 400;">2. Preprocessing Needs: education_level and university_tier should be encoded ordinally to preserve their inherent rank.
    <li style="
        color: #94a3b8;
        margin: 5px 0 0 0;
        font-size: 1.2rem;
        font-weight: 400;">3. Model Choice: Given the mix of categorical and numerical data, a tree-based ensemble (like Random Forest or XGBoost) is ideal for maximizing accuracy.
</p>

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">Feature Engineering and Modeling ⚡</span></div>

In [ ]:
#1. Preprocess
df['education_level'] = df['education_level'].map({'Bachelors': 1, 'Masters': 2, 'PhD': 3})
df['university_tier'] = df['university_tier'].map({'Tier 1': 3, 'Tier 2': 2, 'Tier 3': 1})
df = pd.get_dummies(df, columns=['company_type'])

# 2. Split Data
X = df.drop(['candidate_id', 'hired'], axis=1)
y = df['hired']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Build Robust XGBoost Model
xgb_model = XGBClassifier(
    n_estimators=5000,
    max_depth=7,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    device='cuda',
    eval_metric='auc'
)

xgb_model.fit(X_train, y_train)

# 4. Evaluation
y_pred = xgb_model.predict(X_test)
print(f"XGBoost Accuracy: {accuracy_score(y_test, y_pred):.4f}")

XGBoost Accuracy: 0.7057


<p style="
    color: #94a3b8;
    margin: 5px 0 0 0;
    font-size: 1.2rem;
    font-weight: 400;">
    The baseline model achieved 70.5% accuracy by defaulting to the majority class (Hired). While this accuracy seems acceptable, the Zero Recall for Class 0 renders the model useless for practical recruitment.
</p>

# <div style="    background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);    padding: 25px;    border-radius: 12px;    border-left: 8px solid #00d4ff;    box-shadow: 0 4px 15px rgba(0,0,0,0.3);    margin-bottom: 20px;    font-family: 'Inter', sans-serif;">    <span style="    color: #00d4ff;     margin: 0;    letter-spacing: 2px;     font-weight: 800;    font-size: 1.8rem;">Feature Importance Visualization</span></div>

In [ ]:
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=True)

fig = px.bar(importances, x='Importance', y='Feature', orientation='h',
             title='<b>Feature Importance </b>',
             template='plotly_dark', color='Importance',
             color_continuous_scale='Viridis')
fig.show()